# 02 — Dimension Tables

**Project:** World Forest Change & GDP Correlation Analysis
**Database:** db_forestgdp (MSSQL Server)
**Team:** [Kucerova Kristina], [Kmetova Barbara]
**Date:** May 2026

## Purpose
This notebook builds the dimension tables in the `dbo` schema based on
findings from 01_raw_exploration. All data quality issues identified
in the exploration step are handled here.

## Schema structure
- `raw` — source tables loaded as-is from Kaggle
- `dbo` — cleaned dimension and fact tables

In [2]:
-- Create dim_country
CREATE TABLE dbo.dim_country (
    country_code    VARCHAR(50)     NOT NULL PRIMARY KEY,
    country_name    NVARCHAR(100),
    region          NVARCHAR(100),
    income_group    NVARCHAR(50)
);

Commands completed successfully.

Total execution time: 00:00:00.022

In [3]:
-- Insert all countries that have a valid ISO3 code
INSERT INTO dbo.dim_country (country_code, country_name, region, income_group)
SELECT DISTINCT
    f.Code          AS country_code,
    f.Country       AS country_name,
    i.region,
    i.income_group
FROM raw.Forest_year f
LEFT JOIN raw.stg_gdp_income i ON i.country_code = f.Code
WHERE f.Code IS NOT NULL AND f.Code <> '';

(214 rows affected)

Total execution time: 00:00:00.110

In [5]:
-- Insert entities that have no ISO3 code
-- CONCAT builds AGG_ prefix e.g. AGG_European_Union
INSERT INTO dbo.dim_country (country_code, country_name, region, income_group)
SELECT DISTINCT
    CONCAT('AGG_', REPLACE(f.Country, ' ', '_')) AS country_code,
    f.Country       AS country_name,
    ig.region,
    ig.income_group
FROM raw.Forest_year f
LEFT JOIN raw.stg_gdp_income ig ON ig.country_code = f.Code
WHERE f.Code IS NULL OR f.Code = '';

(7 rows affected)

Total execution time: 00:00:00.040

In [6]:
-- Create dim_year
CREATE TABLE dbo.dim_year (year INT NOT NULL PRIMARY KEY);

Commands completed successfully.

Total execution time: 00:00:00.032

In [7]:
-- Creat dim_year with all years present in Forest_year (the largest date range)
INSERT INTO dbo.dim_year (year)
SELECT DISTINCT Year
FROM raw.Forest_year
ORDER BY Year;

(120 rows affected)

Total execution time: 00:00:00.039